# CO₂ Feedstock Exploration

This notebook companions `how_to_use_demandforge.ipynb`. It audits the new
CO₂ feedstock column that `load_bundle()` now attaches to every row of the
scenario output, and it shows how CO₂ demand *resonates with* — but is
structurally different from — the H₂ demand trajectories.

**Physical foundation** (see `demandforge/process/co2_feedstock.py` and
`demandforge/load_projection/constants.py`):

| Route | Reaction | CO₂ : H₂ (t/t) |
|---|---|---|
| e-methanol | `CO₂ + 3 H₂ → CH₃OH + H₂O` | **7.277** |
| methanol-to-jet | MeOH step governs | **7.277** |
| Fischer-Tropsch | `12 CO₂ + 37 H₂ → C₁₂H₂₆ + 24 H₂O` (RWGS + FT) | **7.080** |
| Sabatier (unused) | `CO₂ + 4 H₂ → CH₄ + 2 H₂O` | 5.458 |
| Haber-Bosch, DRI-H₂, HT, bio-naphtha, chem-recycling | no CO₂ reactant | **0.0** |

Per-pathway H₂ intensity of the eSAF route:

| Pathway | t H₂ / t SAF | Derivation |
|---|---|---|
| `fischer_tropsch` | **0.46** | floor 0.438 × 1.05 (H₂ slip + oligo HT) |
| `methanol_to_jet` | **0.57** | floor 0.438 / 0.77 (MeOH × MTO × HT cascade C-yield) |

Both above the stoichiometric floor 0.4380 t H₂/t SAF from the C₁₂H₂₆ balance.

## 0 — Setup and scenario load

Run every bundle once and keep the frames in `all_scenarios`. Each row
carries both `h2_demand_t_per_yr` and `co2_demand_t_per_yr`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from demandforge.load_projection.scenarios import (
    list_bundles, get_bundle_params, load_bundle,
)

SCENARIO_COLOURS = {
    "low_h2":           "#a8dadc",
    "central":          "#457b9d",
    "high_h2":          "#e63946",
    "industry_stress":  "#1d3557",
}
SECTOR_COLOURS = {
    "steel":    "#264653",
    "refinery": "#2a9d8f",
    "ammonia":  "#8ecae6",
    "maritime": "#457b9d",
    "olefins":  "#e9c46a",
    "esaf":     "#f4a261",
}
ordered_sectors = ["steel", "refinery", "ammonia", "maritime", "olefins", "esaf"]

bundles = list_bundles()
all_scenarios = {name: load_bundle(name) for name in bundles}
# Sanity: confirm the new CO2 column is present
assert "co2_demand_t_per_yr" in all_scenarios["central"].columns, (
    "co2_demand_t_per_yr missing — did you forget to pull the new co2_feedstock module?"
)
print("Bundles loaded:", list(all_scenarios))
print("Central frame columns:", list(all_scenarios['central'].columns))

## 1 — EU-27 totals at 2050: H₂ vs CO₂ per bundle

First-glance resonance check. The CO₂/H₂ ratio here is a *bundle-level*
ratio — NOT a universal constant, because ammonia, refinery and steel
contribute zero CO₂ charge while consuming significant H₂.

In [ ]:
summary_rows = []
for bname, df_b in all_scenarios.items():
    df_50 = df_b[df_b["year"] == 2050]
    h2_mt  = df_50["h2_demand_t_per_yr"].sum()  / 1e6
    co2_mt = df_50["co2_demand_t_per_yr"].sum() / 1e6
    by_sec_co2 = df_50.groupby("sector")["co2_demand_t_per_yr"].sum() / 1e6
    summary_rows.append({
        "bundle":   bname,
        "H2_Mt":    round(h2_mt,  3),
        "CO2_Mt":   round(co2_mt, 3),
        "CO2/H2":   round(co2_mt / h2_mt, 3) if h2_mt else 0.0,
        "CO2_maritime_Mt": round(by_sec_co2.get("maritime", 0.0), 3),
        "CO2_olefins_Mt":  round(by_sec_co2.get("olefins",  0.0), 3),
        "CO2_esaf_Mt":     round(by_sec_co2.get("esaf",     0.0), 3),
    })
summary = pd.DataFrame(summary_rows).set_index("bundle")
summary

## 2 — Dual-axis trajectory: H₂ demand vs CO₂ feedstock (central bundle)

Time-resolved resonance. Both curves should grow together, but with the
CO₂ curve scaled ~2.8–3.4× the H₂ curve (ratio depends on sector mix).

In [ ]:
df_c = all_scenarios["central"]
ts = (df_c.groupby("year")
          [["h2_demand_t_per_yr", "co2_demand_t_per_yr"]]
          .sum() / 1e6)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(ts.index, ts["h2_demand_t_per_yr"],  color="#1d3557", lw=2.5, label="H₂ demand")
ax2.plot(ts.index, ts["co2_demand_t_per_yr"], color="#e63946", lw=2.5,
         ls="--", label="CO₂ feedstock")
ax1.set_ylabel("H₂ (Mt/yr)", color="#1d3557")
ax2.set_ylabel("CO₂ feedstock (Mt/yr)", color="#e63946")
ax1.set_xlim(2019, 2050)
ax1.set_title("EU-27 central bundle — H₂ demand vs CO₂ feedstock trajectory")
ax1.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 3 — Sector decomposition of CO₂ feedstock at 2050

Only three sectors consume CO₂ as feedstock: **maritime** (e-methanol),
**olefins** (MTO), **eSAF** (FT + MtJ). The other three are zero by
construction. eSAF typically dominates by an order of magnitude.

In [ ]:
co2_sector = {}
for bname, df_b in all_scenarios.items():
    df_50 = df_b[df_b["year"] == 2050]
    co2_sector[bname] = (
        df_50.groupby("sector")["co2_demand_t_per_yr"].sum() / 1e6
    )
co2_pivot = pd.DataFrame(co2_sector).fillna(0)
print("CO₂ feedstock (Mt/yr) by sector × bundle at 2050:")
display(co2_pivot.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
co2_pivot.T[[s for s in ordered_sectors if s in co2_pivot.index]].plot.bar(
    ax=ax, stacked=True, width=0.68, edgecolor="white",
    color=[SECTOR_COLOURS[s] for s in ordered_sectors if s in co2_pivot.index],
)
ax.set_ylabel("CO₂ feedstock (Mt/yr)")
ax.set_xlabel("")
ax.set_title("CO₂ feedstock demand at 2050 — by sector and bundle")
ax.legend(title="Sector", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4 — Four-bundle H₂ and CO₂ comparison — total EU-27 trajectory

Same story as the notebook's §7 "EU-27 total industrial H₂ demand", but
paired with the CO₂ counterpart. The bundle ordering is preserved under
CO₂ — `low_h2 < central < high_h2 < industry_stress` — but the spread
widens because eSAF (the dominant CO₂ consumer) has more dynamic range
across scenarios.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for bname, colour in SCENARIO_COLOURS.items():
    df_b = all_scenarios[bname]
    h2_ts  = df_b.groupby("year")["h2_demand_t_per_yr"].sum()  / 1e6
    co2_ts = df_b.groupby("year")["co2_demand_t_per_yr"].sum() / 1e6
    axL.plot(h2_ts.index, h2_ts.values, color=colour, lw=2.5, label=bname)
    axR.plot(co2_ts.index, co2_ts.values, color=colour, lw=2.5, label=bname)

axL.set_title("EU-27 H₂ demand (Mt/yr)")
axR.set_title("EU-27 CO₂ feedstock demand (Mt/yr)")
for ax in (axL, axR):
    ax.grid(alpha=0.25)
    ax.set_xlim(2019, 2050)
axL.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## 5 — Pathway-mix coherence check (single-country, eSAF sector only)

The key physical invariant introduced by the refactor: shifting the FT ↔
MtJ mix must move *both* H₂ and CO₂ together, because MtJ has BOTH a
higher H₂ intensity (0.57 vs 0.46 t H₂/t SAF) AND a higher CO₂:H₂ ratio
(7.277 vs 7.080 t CO₂/t H₂).

For FR, eSAF-only, the bundle-to-bundle variation should be monotone in
the MtJ share (low_h2 → central → high_h2 → industry_stress = 0 → 30 %
→ 60 % → 70 %).

In [ ]:
cc = "FR"
rows = []
for bname, df_b in all_scenarios.items():
    df_50 = df_b[(df_b["year"] == 2050) &
                 (df_b["country"] == cc) &
                 (df_b["sector"]  == "esaf")]
    if df_50.empty:
        continue
    params = get_bundle_params(bname)
    ps = params["esaf"].get("pathway_shares", {"fischer_tropsch": 1.0, "methanol_to_jet": 0.0})
    rows.append({
        "bundle":       bname,
        "FT_share":     ps.get("fischer_tropsch", 0.0),
        "MtJ_share":    ps.get("methanol_to_jet", 0.0),
        "H2_kt":        float(df_50["h2_demand_t_per_yr"].sum()  / 1e3),
        "CO2_kt":       float(df_50["co2_demand_t_per_yr"].sum() / 1e3),
    })
coh = pd.DataFrame(rows).set_index("bundle")
coh["CO2/H2"] = coh["CO2_kt"] / coh["H2_kt"]
print(f"{cc} eSAF-only at 2050 — joint response to FT/MtJ mix:")
display(coh.round(3))

# Monotonicity assertion — H2 and CO2 must both rise with MtJ share
coh_sorted = coh.sort_values("MtJ_share")
assert (coh_sorted["H2_kt"].diff().dropna() >= -1e-9).all(),  "H2 not monotone in MtJ share"
assert (coh_sorted["CO2_kt"].diff().dropna() >= -1e-9).all(), "CO2 not monotone in MtJ share"
print("Monotonicity OK — both H2 and CO2 rise jointly with MtJ share.")

## 6 — Scatter: H₂ vs CO₂ per (country, sector, bundle) at 2050

Each point is one (country, sector, bundle) combination. The two dashed
reference lines show the stoichiometric ceilings: 7.277 (e-methanol / MtJ)
and 7.080 (Fischer-Tropsch). Every point should lie *on or between* these
lines for CO₂-consuming sectors — a graphical mass-balance check.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
CO2_SECTORS = ("maritime", "olefins", "esaf")
colours = {"maritime": "#457b9d", "olefins": "#2a9d8f", "esaf": "#f4a261"}

labels_drawn = set()
for bname, df_b in all_scenarios.items():
    df_50 = df_b[(df_b["year"] == 2050) &
                 (df_b["sector"].isin(CO2_SECTORS)) &
                 (df_b["h2_demand_t_per_yr"] > 0)]
    for sec in CO2_SECTORS:
        sub = df_50[df_50["sector"] == sec]
        lbl = sec if sec not in labels_drawn else None
        labels_drawn.add(sec)
        ax.scatter(
            sub["h2_demand_t_per_yr"]  / 1e3,
            sub["co2_demand_t_per_yr"] / 1e3,
            label=lbl, color=colours[sec], alpha=0.55, s=30,
        )

# Reference ceilings
h2_max = ax.get_xlim()[1]
xs = np.linspace(0, h2_max, 100)
ax.plot(xs, xs * 7.277, "k--", alpha=0.40, lw=1, label="7.277 (eMeOH / MtJ)")
ax.plot(xs, xs * 7.080, "k:",  alpha=0.40, lw=1, label="7.080 (Fischer-Tropsch)")
ax.set_xlabel("H₂ demand (kt/yr)")
ax.set_ylabel("CO₂ feedstock (kt/yr)")
ax.set_title("H₂ ↔ CO₂ coherence at 2050 — each dot = (country, sector, bundle)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

## 7 — Stacked-area CO₂ trajectory per bundle

The CO₂ analogue of the notebook's §22 sector-composition stackplots.
Only maritime, olefins, eSAF contribute; eSAF typically explains 80–90 %
of the total post-2035.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)

for ax, (bname, _) in zip(axes.flatten(), SCENARIO_COLOURS.items()):
    df_b = all_scenarios[bname]
    pivot = df_b.pivot_table(
        index="year", columns="sector",
        values="co2_demand_t_per_yr", aggfunc="sum", fill_value=0,
    ) / 1e6  # Mt CO2/yr
    # Keep only CO2-consuming sectors, in consistent order
    keep = [s for s in ("maritime", "olefins", "esaf") if s in pivot.columns]
    pivot = pivot[keep]
    ax.stackplot(
        pivot.index, *[pivot[s] for s in pivot.columns],
        labels=[s.capitalize() for s in pivot.columns],
        colors=[SECTOR_COLOURS[s] for s in pivot.columns],
        alpha=0.85,
    )
    ax.set_title(f"CO₂ feedstock — {bname}")
    ax.set_ylabel("Mt CO₂ / yr")
    ax.set_xlim(2019, 2050)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=8, frameon=False)

fig.suptitle("EU-27 CO₂ feedstock demand — stacked by sector, four bundles",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 8 — Per-pathway reconstruction audit (FR, 2050, eSAF)

End-to-end integrity check: does the `co2_demand_t_per_yr` attached by
`load_bundle` match the physical reconstruction from per-pathway H₂
columns × stoichiometric ratios? Should match to 1e-9.

In [ ]:
# End-to-end mass-balance audit.
#
# The audit must REPLICATE the load_bundle preprocessing pipeline,
# otherwise the upstream refinery throughput (and therefore the jet-fuel
# pool that drives eSAF demand) will diverge from what the bundle saw.
# Specifically, load_bundle applies, in order:
#   1. apply_capacity_delay( capacity_delay_years )
#   2. get_naphtha_for_crackers(...)  -> petrochem naphtha supply
#   3. apply_petrochem_naphtha_correction( PETROCHEM_NAPHTHA_FRACTION )
# BEFORE handing units_config to project_refinery_h2_demand.  Skipping
# either of those preprocessing steps shifts the refinery's jet output
# and breaks any downstream reconstruction.

from demandforge.load_projection.hydrogen import (
    project_refinery_h2_demand, project_esaf_h2_demand,
)
from demandforge.load_projection.constants import (
    PETROCHEM_NAPHTHA_FRACTION, STEAM_CRACKER_OLEFIN_YIELD,
)
from demandforge.process.refinery import (
    CONCAWE_MORE_MOLECULE, CONCAWE_MAX_ELECTRON,
    apply_capacity_delay,
    apply_petrochem_naphtha_correction,
    get_naphtha_for_crackers,
)
from demandforge.process.esaf import DEFAULT_JET_UNIT_NAMES, DEFAULT_JET_YIELDS
from demandforge.process.co2_feedstock import CO2_STOICHIOMETRY_T_PER_T_H2

BUNDLE_NAME = "central"
COUNTRY     = "FR"
REF_YEAR    = 2019
TGT_YEAR    = 2050

params = get_bundle_params(BUNDLE_NAME)
ref_pars = params["refinery"]

# 1. Pick the right CONCAWE config based on molecule_scenario
mol_scenario = ref_pars.get("molecule_scenario", "more-molecule")
units_config = (CONCAWE_MAX_ELECTRON if mol_scenario == "max-electron"
                else CONCAWE_MORE_MOLECULE)

# 2. Apply capacity delay (load_bundle does this BEFORE petrochem correction)
delay_years = ref_pars.get("capacity_delay_years", 0)
if delay_years > 0:
    units_config = apply_capacity_delay(
        units_config, delay_years=delay_years, reference_year=REF_YEAR,
    )

# 3. Apply petrochem naphtha correction (load_bundle does this whenever
#    olefins is in the sector list, which is the default)
years = np.arange(REF_YEAR, TGT_YEAR + 1)
_ = get_naphtha_for_crackers(
    units_config, years, petrochem_fraction=PETROCHEM_NAPHTHA_FRACTION,
)
units_config = apply_petrochem_naphtha_correction(
    units_config, petrochem_fraction=PETROCHEM_NAPHTHA_FRACTION,
)

# 4. Refinery → eSAF, mirroring load_bundle's piping
df_ref = project_refinery_h2_demand(
    country=[COUNTRY], reference_year=REF_YEAR, target_year=TGT_YEAR,
    units_config=units_config,
    inefficiency_share=ref_pars["inefficiency_share"],
)

esaf_df = project_esaf_h2_demand(
    country=[COUNTRY], reference_year=REF_YEAR, target_year=TGT_YEAR,
    demand_growth_rate=params["esaf"]["demand_growth_rate"],
    pathway_shares=params["esaf"]["pathway_shares"],
    refinery_df=df_ref,
    jet_unit_names=DEFAULT_JET_UNIT_NAMES,
    jet_yields=DEFAULT_JET_YIELDS,
)

# 5. Per-pathway CO2 reconstruction
col_ft  = "h2_demand_for_esaf_fischer_tropsch_t_per_yr"
col_mtj = "h2_demand_for_esaf_methanol_to_jet_t_per_yr"

manual = (
    esaf_df[col_ft]  * CO2_STOICHIOMETRY_T_PER_T_H2["fischer_tropsch"]
    + esaf_df[col_mtj] * CO2_STOICHIOMETRY_T_PER_T_H2["methanol_to_jet"]
)
manual.index = esaf_df["year"]

attached = (
    df_c[(df_c["sector"] == "esaf") & (df_c["country"] == COUNTRY)]
    .set_index("year")["co2_demand_t_per_yr"]
)

# Cross-check H2 too — if H2 differs the audit is useless
h2_manual   = (esaf_df.set_index("year")[col_ft]
               + esaf_df.set_index("year")[col_mtj])
h2_attached = (df_c[(df_c["sector"] == "esaf") & (df_c["country"] == COUNTRY)]
               .set_index("year")["h2_demand_t_per_yr"])
h2_err_max  = (h2_manual - h2_attached).abs().max()

cmp = pd.concat([manual.rename("manual"), attached.rename("attached")],
                axis=1).dropna()
cmp["abs_err"] = (cmp["manual"] - cmp["attached"]).abs()
cmp["rel_err"] = cmp["abs_err"] / cmp["attached"].replace(0, np.nan)

print(f"{COUNTRY} eSAF — physical reconstruction vs attached column (last 5 years):")
display(cmp.tail().round(4))
print(f"H2  max abs error across all years: {h2_err_max:.2e} t  "
      f"(must be ~0 if preprocessing matches load_bundle)")
print(f"CO2 max abs error across all years: {cmp['abs_err'].max():.2e} t  "
      f"(rel: {cmp['rel_err'].max():.2e})")

# Tolerance: 1 t CO2 absolute is well within float roundoff for ~1e7 t totals
TOL = 1.0
assert h2_err_max < TOL, (
    f"H2 reconstruction mismatch ({h2_err_max:.2e} t).  "
    "load_bundle preprocessing not faithfully replicated."
)
assert cmp["abs_err"].max() < TOL, (
    f"CO2 reconstruction mismatch ({cmp['abs_err'].max():.2e} t).  "
    "Per-pathway H2 × stoichiometric ratio does not equal the attached column."
)
print("OK — per-pathway reconstruction matches the attached column "
      f"within {TOL:.0e} t CO2.")

## 10 — Pathway-mix Pareto frontier (volume-isolated)

The §5 bundle scan confounds two levers — eSAF *production volume* and
*pathway mix*. To isolate the mix lever, hold SAF production constant
at 1 Mt/yr and sweep MtJ share from 0 → 100 %. The result is the
analytic locus of feasible (H₂, CO₂) pairs per t SAF:

    H₂(s)  = (1−s)·0.46 + s·0.57         t H₂  / t SAF
    CO₂(s) = (1−s)·0.46·7.080 + s·0.57·7.277   t CO₂ / t SAF

The four bundle endpoints (FR-only at 2050, normalised by their SAF
output) are over-plotted; they should land *exactly* on the analytic
line — a model-vs-formula closure check.

In [ ]:
from demandforge.load_projection.constants import H2_INTENSITY_T_PER_T_ESAF_BY_PATHWAY
from demandforge.process.co2_feedstock import CO2_STOICHIOMETRY_T_PER_T_H2

i_ft, i_mtj = (H2_INTENSITY_T_PER_T_ESAF_BY_PATHWAY["fischer_tropsch"],
               H2_INTENSITY_T_PER_T_ESAF_BY_PATHWAY["methanol_to_jet"])
r_ft, r_mtj = (CO2_STOICHIOMETRY_T_PER_T_H2["fischer_tropsch"],
               CO2_STOICHIOMETRY_T_PER_T_H2["methanol_to_jet"])

s = np.linspace(0.0, 1.0, 101)
h2_per_saf  = (1 - s) * i_ft           + s * i_mtj
co2_per_saf = (1 - s) * i_ft * r_ft    + s * i_mtj * r_mtj
ratio       = co2_per_saf / h2_per_saf  # blended CO2:H2 (effective)

# Bundle endpoints (FR, eSAF, 2050) — recover SAF production from H2 split
endpoints = []
for bname in ["low_h2", "central", "high_h2", "industry_stress"]:
    df_b = all_scenarios[bname]
    sub  = df_b[(df_b["year"] == 2050) &
                (df_b["country"] == "FR") &
                (df_b["sector"]  == "esaf")]
    if sub.empty:
        continue
    pp = get_bundle_params(bname)["esaf"].get("pathway_shares", {})
    s_b = pp.get("methanol_to_jet", 0.0)
    h2_b  = float(sub["h2_demand_t_per_yr"].sum())
    co2_b = float(sub["co2_demand_t_per_yr"].sum())
    saf_b = h2_b / ((1 - s_b) * i_ft + s_b * i_mtj)
    endpoints.append({"bundle": bname, "s_MtJ": s_b,
                      "h2_per_saf":  h2_b  / saf_b,
                      "co2_per_saf": co2_b / saf_b})
end_df = pd.DataFrame(endpoints)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: H2 and CO2 per t SAF as a function of MtJ share
axA = axes[0]
axA2 = axA.twinx()
axA.plot(s, h2_per_saf,  color="#1d3557", lw=2.5, label="H₂  (left axis)")
axA2.plot(s, co2_per_saf, color="#e63946", lw=2.5, ls="--", label="CO₂ (right axis)")
axA.set_xlabel("MtJ share (s)")
axA.set_ylabel("t H₂ / t SAF",  color="#1d3557")
axA2.set_ylabel("t CO₂ / t SAF", color="#e63946")
axA.set_title("A. Per-t-SAF intensities vs MtJ share")
axA.grid(alpha=0.25)

# Panel B: Pareto locus in (H2, CO2) space + bundle endpoints
axB = axes[1]
axB.plot(h2_per_saf, co2_per_saf, color="#264653", lw=2.5,
         label="Analytic locus (s ∈ [0,1])")
for _, row in end_df.iterrows():
    axB.scatter(row["h2_per_saf"], row["co2_per_saf"],
                s=85, color=SCENARIO_COLOURS[row["bundle"]],
                edgecolor="black", linewidth=0.6, zorder=5,
                label=f"{row['bundle']} (s={row['s_MtJ']:.2f})")
axB.set_xlabel("t H₂ / t SAF")
axB.set_ylabel("t CO₂ / t SAF")
axB.set_title("B. Pareto frontier — bundle endpoints overlaid")
axB.grid(alpha=0.25)
axB.legend(fontsize=8, frameon=False, loc="lower right")

# Panel C: effective CO2:H2 ratio along the locus (always between FT and MtJ stoich)
axC = axes[2]
axC.plot(s, ratio, color="#2a9d8f", lw=2.5)
axC.axhline(r_ft,  color="#1d3557", ls="--", lw=1, alpha=0.6,
            label=f"FT stoich {r_ft:.3f}")
axC.axhline(r_mtj, color="#e63946", ls=":",  lw=1, alpha=0.6,
            label=f"MtJ stoich {r_mtj:.3f}")
axC.set_xlabel("MtJ share (s)")
axC.set_ylabel("Effective CO₂ / H₂  (t/t)")
axC.set_title("C. Blended CO₂:H₂ ratio")
axC.grid(alpha=0.25)
axC.legend(fontsize=8, frameon=False)

fig.tight_layout()
plt.show()

# Quantify the closure check
end_df["co2_analytic"] = (
    (1 - end_df["s_MtJ"]) * i_ft * r_ft
    + end_df["s_MtJ"] * i_mtj * r_mtj
)
end_df["err_pct"] = 100 * (end_df["co2_per_saf"] - end_df["co2_analytic"]) / end_df["co2_analytic"]
print("Bundle-endpoint closure (must be ~0 %):")
display(end_df[["bundle", "s_MtJ", "h2_per_saf", "co2_per_saf",
                "co2_analytic", "err_pct"]].round(4))

## 11 — EU-27 geographic map of CO₂ feedstock at 2050

Bubble map placing each country at its capital-city centroid, bubble
**area** ∝ total CO₂ feedstock demand (Mt/yr at 2050), **colour** = the
dominant CO₂-consuming sector for that country. Four panels — one per
bundle — share the same area scale so that countries are directly
comparable across scenarios.

The map deliberately uses no shapefile dependency: a hardcoded EU-27
capital-city lat/lon dictionary is the only geographic input, which
keeps the notebook reproducible in any base scientific Python env.

In [ ]:
# Capital-city centroids (lat, lon) for EU-27.  Reasonable proxy when no
# shapefile is available — accurate enough for visual ranking, never used
# for any quantitative calculation.
EU27_CENTROIDS = {
    "AT": (48.21, 16.37), "BE": (50.85,  4.35), "BG": (42.70, 23.32),
    "CY": (35.18, 33.38), "CZ": (50.08, 14.44), "DE": (52.52, 13.40),
    "DK": (55.68, 12.57), "EE": (59.44, 24.75), "ES": (40.42, -3.70),
    "FI": (60.17, 24.94), "FR": (48.86,  2.35), "GR": (37.98, 23.73),
    "HR": (45.81, 15.98), "HU": (47.50, 19.04), "IE": (53.35, -6.26),
    "IT": (41.90, 12.50), "LT": (54.69, 25.28), "LU": (49.61,  6.13),
    "LV": (56.95, 24.11), "MT": (35.90, 14.51), "NL": (52.37,  4.90),
    "PL": (52.23, 21.01), "PT": (38.72, -9.14), "RO": (44.43, 26.10),
    "SE": (59.33, 18.07), "SI": (46.06, 14.51), "SK": (48.15, 17.11),
}

CO2_SECTORS = ["maritime", "olefins", "esaf"]
SECTOR_MAP_COLOUR = {"maritime": "#457b9d", "olefins": "#2a9d8f", "esaf": "#f4a261"}

def _country_co2_2050(df_b: pd.DataFrame) -> pd.DataFrame:
    sub = df_b[(df_b["year"] == 2050) & (df_b["sector"].isin(CO2_SECTORS))]
    by_country = (sub.groupby(["country", "sector"])["co2_demand_t_per_yr"]
                     .sum().unstack(fill_value=0.0) / 1e6)  # Mt
    by_country["total"]    = by_country.sum(axis=1)
    by_country["dominant"] = by_country[CO2_SECTORS].idxmax(axis=1)
    return by_country

# Build per-bundle frames and a global max for shared area scale
country_frames = {b: _country_co2_2050(df) for b, df in all_scenarios.items()}
global_max_mt  = max(f["total"].max() for f in country_frames.values())

def _area_to_marker(mt: float, max_mt: float) -> float:
    # Marker area (s in scatter) scales linearly with Mt; cap at 1500 pt²
    return float(np.clip(1500.0 * mt / max_mt, 6.0, 1500.0))

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, bname in zip(axes.flatten(),
                     ["low_h2", "central", "high_h2", "industry_stress"]):
    f = country_frames[bname]
    for cc, (lat, lon) in EU27_CENTROIDS.items():
        if cc not in f.index:
            continue
        mt   = float(f.loc[cc, "total"])
        if mt < 1e-3:
            continue
        dom  = f.loc[cc, "dominant"]
        ax.scatter(lon, lat, s=_area_to_marker(mt, global_max_mt),
                   color=SECTOR_MAP_COLOUR[dom], alpha=0.65,
                   edgecolor="black", linewidth=0.5, zorder=3)
        # Label only the largest 8 to keep map readable
        ax.text(lon, lat, cc, ha="center", va="center", fontsize=7,
                fontweight="bold", color="white", zorder=4)
    ax.set_xlim(-12, 32)
    ax.set_ylim(34, 62)
    ax.set_aspect(1.4)         # crude lat/lon distortion correction at ~50°N
    ax.set_title(f"{bname}  —  {f['total'].sum():.1f} Mt CO₂/yr (2050)",
                 fontsize=11)
    ax.grid(alpha=0.20)
    ax.set_xticks([]); ax.set_yticks([])

# Legend swatches
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", label=s.capitalize(),
                  markerfacecolor=c, markeredgecolor="black",
                  markersize=11, alpha=0.7)
           for s, c in SECTOR_MAP_COLOUR.items()]
# Bubble-size legend
size_anchors_mt = [1, 5, 10, 20]
size_handles = [Line2D([0], [0], marker="o", color="w",
                       label=f"{m} Mt CO₂/yr",
                       markerfacecolor="#888888", markeredgecolor="black",
                       markersize=np.sqrt(_area_to_marker(m, global_max_mt)),
                       alpha=0.65)
                for m in size_anchors_mt]
fig.legend(handles=handles, loc="lower left",  bbox_to_anchor=(0.02, 0.00),
           frameon=False, title="Dominant sector")
fig.legend(handles=size_handles, loc="lower right", bbox_to_anchor=(0.98, 0.00),
           frameon=False, title="Total CO₂ demand", ncol=4)

fig.suptitle("EU-27 CO₂ feedstock demand at 2050 — geographic distribution",
             fontsize=13, y=0.995)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
plt.show()

## 12 — Country × year heatmap of CO₂ feedstock (central bundle)

Where and when does the EU-27 CO₂ requirement materialise? Heatmap of
the top-15 countries (rows) × year (columns), value = total CO₂
feedstock demand. Diagonal-ish growth bands reveal the staggered
build-out of e-fuel supply chains country by country.

In [ ]:
df_b = all_scenarios["central"]
hmap = (df_b[df_b["sector"].isin(CO2_SECTORS)]
        .pivot_table(index="country", columns="year",
                     values="co2_demand_t_per_yr",
                     aggfunc="sum", fill_value=0.0) / 1e6)  # Mt CO2/yr

# Top 15 countries by 2050 demand
top15 = hmap[2050].sort_values(ascending=False).head(15).index.tolist()
hmap = hmap.loc[top15]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(hmap.values, aspect="auto", cmap="YlOrRd",
               interpolation="nearest")
ax.set_xticks(range(0, len(hmap.columns), 5))
ax.set_xticklabels(hmap.columns[::5])
ax.set_yticks(range(len(hmap.index)))
ax.set_yticklabels(hmap.index)
ax.set_xlabel("Year")
ax.set_title("Central bundle — CO₂ feedstock demand (Mt/yr) "
             "by country × year (top-15 countries)")
cb = fig.colorbar(im, ax=ax, shrink=0.85)
cb.set_label("Mt CO₂ / yr")
plt.tight_layout()
plt.show()

# Concentration metric — Herfindahl-Hirschman Index of the 2050 distribution
shares_2050 = (hmap[2050] / hmap[2050].sum()).values
HHI = float((shares_2050 ** 2).sum())
top3_share = float(hmap[2050].nlargest(3).sum() / hmap[2050].sum())
print(f"2050 country concentration — HHI = {HHI:.3f}, "
      f"top-3 share = {top3_share:.1%}  "
      "(HHI > 0.25 → concentrated; > 0.18 → moderately concentrated; "
      "< 0.10 → diffuse).")

## 13 — Country × sector treemap at 2050 (central bundle)

A space-filling treemap uses cell **area** ∝ CO₂ feedstock demand and
cell **colour** ∝ sector. Outer cells = countries (ranked); inner cells
= the maritime/olefins/eSAF split inside each country. This is the most
information-dense single view of the 2050 CO₂ landscape: at a glance
you see *who* drives EU CO₂ demand and *which sector* dominates inside
each country. No external dependency — a recursive squarified-treemap
layout is implemented inline.

In [ ]:
# Squarified treemap layout — Bruls, Huijing, van Wijk (2000), simplified.
# Returns a list of (x, y, w, h) rectangles whose areas equal the input
# values, packed into the bounding rectangle (X, Y, W, H).
def _squarify(values, X, Y, W, H):
    if not values:
        return []
    if len(values) == 1:
        return [(X, Y, W, H)]
    total = sum(values)
    # Place along the shorter side first
    if W >= H:
        # vertical strips
        x_acc = X
        recs = []
        for v in values:
            w = W * v / total
            recs.append((x_acc, Y, w, H))
            x_acc += w
        return recs
    else:
        y_acc = Y
        recs = []
        for v in values:
            h = H * v / total
            recs.append((X, y_acc, W, h))
            y_acc += h
        return recs

# Country totals at 2050 (central), then sector breakdown inside each country
df_b = all_scenarios["central"]
sub = df_b[(df_b["year"] == 2050) & (df_b["sector"].isin(CO2_SECTORS))]
country_total = (sub.groupby("country")["co2_demand_t_per_yr"].sum() / 1e6)
country_total = country_total[country_total > 0.05].sort_values(ascending=False)

# Recursive split: alternate orientation by aspect ratio
fig, ax = plt.subplots(figsize=(13, 8))

# Outer layout — countries
W, H = 100.0, 100.0  # arbitrary canvas
country_recs = _squarify(country_total.tolist(), 0, 0, W, H)

for cc, (x, y, w, h) in zip(country_total.index, country_recs):
    # Inner layout — sectors inside this country
    inner = (sub[sub["country"] == cc]
             .groupby("sector")["co2_demand_t_per_yr"].sum() / 1e6)
    inner = inner.reindex(CO2_SECTORS, fill_value=0.0)
    inner_recs = _squarify(inner.tolist(), x, y, w, h)
    for sec, (ix, iy, iw, ih) in zip(CO2_SECTORS, inner_recs):
        if iw <= 0 or ih <= 0:
            continue
        ax.add_patch(plt.Rectangle((ix, iy), iw, ih,
                                   facecolor=SECTOR_MAP_COLOUR[sec],
                                   edgecolor="white", linewidth=1.0,
                                   alpha=0.92))
    # Country label sized by area
    if w * h > 60:
        fs = max(7, min(13, int(0.6 * np.sqrt(w * h))))
        ax.text(x + w / 2, y + h / 2, cc, ha="center", va="center",
                fontsize=fs, fontweight="bold", color="black")

ax.set_xlim(0, W); ax.set_ylim(0, H)
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("CO₂ feedstock demand at 2050 — country × sector treemap "
             f"(central bundle, total {country_total.sum():.1f} Mt CO₂/yr)")

# Sector colour legend
handles = [plt.Rectangle((0, 0), 1, 1,
                          facecolor=SECTOR_MAP_COLOUR[s], edgecolor="white",
                          alpha=0.92, label=s.capitalize())
           for s in CO2_SECTORS]
ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.18, 1.0),
          frameon=False, title="Sector")
plt.tight_layout()
plt.show()

## 14 — Shift-share decomposition of bundle-to-bundle CO₂ spread

Each bundle's eSAF CO₂ demand at 2050 differs from `central` for two
distinct reasons: a **volume effect** (more or less SAF produced) and
an **intensity / mix effect** (different FT/MtJ blend changing the
t CO₂ per t SAF). A first-order Laspeyres-style decomposition writes:

    ΔCO₂  ≈  Δ(SAF) · I_central    +    SAF_central · ΔI    +    Δ(SAF) · ΔI

with `I = (1−s)·i_FT·r_FT + s·i_MtJ·r_MtJ`. The cross term is small but
not zero — it is shown explicitly so the table sums exactly.

In [ ]:
# EU-27 eSAF SAF production per bundle, recovered from per-pathway H2:
#   SAF = h2_total / [(1-s)*i_ft + s*i_mtj]
# CO2 intensity (t CO2 / t SAF) computed from pathway shares.
def _bundle_esaf_aggregate(bname):
    df_b = all_scenarios[bname]
    sub  = df_b[(df_b["year"] == 2050) & (df_b["sector"] == "esaf")]
    h2_total  = float(sub["h2_demand_t_per_yr"].sum())
    co2_total = float(sub["co2_demand_t_per_yr"].sum())
    pp = get_bundle_params(bname)["esaf"].get("pathway_shares", {})
    s_b = pp.get("methanol_to_jet", 0.0)
    h2_per_saf  = (1 - s_b) * i_ft + s_b * i_mtj
    co2_per_saf = (1 - s_b) * i_ft * r_ft + s_b * i_mtj * r_mtj
    saf_total   = h2_total / h2_per_saf
    return {"bundle": bname, "s_MtJ": s_b,
            "SAF_Mt": saf_total / 1e6,
            "I_co2_per_saf": co2_per_saf,
            "CO2_Mt_observed": co2_total / 1e6,
            "CO2_Mt_check":    saf_total * co2_per_saf / 1e6}

agg = pd.DataFrame([_bundle_esaf_aggregate(b)
                    for b in ["central", "low_h2", "high_h2", "industry_stress"]])
agg["closure_err_Mt"] = agg["CO2_Mt_observed"] - agg["CO2_Mt_check"]
print("Per-bundle aggregates (closure must be ~0):")
display(agg.round(4))

# Decomposition vs central
ref = agg[agg["bundle"] == "central"].iloc[0]
SAF_ref, I_ref, CO2_ref = ref["SAF_Mt"], ref["I_co2_per_saf"], ref["CO2_Mt_observed"]

decomp_rows = []
for _, row in agg.iterrows():
    if row["bundle"] == "central":
        continue
    dSAF = row["SAF_Mt"] - SAF_ref
    dI   = row["I_co2_per_saf"] - I_ref
    vol_effect    = dSAF * I_ref          # Mt CO2
    inten_effect  = SAF_ref * dI           # Mt CO2
    cross_effect  = dSAF * dI              # Mt CO2
    total         = vol_effect + inten_effect + cross_effect
    observed      = row["CO2_Mt_observed"] - CO2_ref
    decomp_rows.append({
        "bundle":         row["bundle"],
        "ΔSAF_Mt":        dSAF,
        "ΔI_t_per_t":     dI,
        "Volume_effect":  vol_effect,
        "Mix_effect":     inten_effect,
        "Cross_effect":   cross_effect,
        "Reconstructed":  total,
        "Observed":       observed,
        "Mix_share_%":    100 * inten_effect / observed if observed else np.nan,
    })
decomp = pd.DataFrame(decomp_rows).set_index("bundle")
print("Shift-share of ΔCO₂ relative to `central` (Mt CO₂/yr at 2050):")
display(decomp.round(3))

# Visual: stacked bars showing how each bundle's gap vs central decomposes
fig, ax = plt.subplots(figsize=(10, 4.5))
x  = np.arange(len(decomp))
ax.bar(x - 0.20, decomp["Volume_effect"], 0.18, color="#457b9d", label="Volume")
ax.bar(x + 0.00, decomp["Mix_effect"],    0.18, color="#e63946", label="Mix")
ax.bar(x + 0.20, decomp["Cross_effect"],  0.18, color="#888888", label="Cross")
ax.scatter(x, decomp["Observed"], marker="D", color="black", zorder=5,
           label="Observed Δ vs central")
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(decomp.index)
ax.set_ylabel("Δ CO₂ vs central, Mt/yr at 2050")
ax.set_title("Shift-share decomposition — eSAF CO₂ feedstock at 2050")
ax.grid(alpha=0.25, axis="y")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## 15 — Supply-chain ramp: peak annual CO₂ build-out & infrastructure equivalents

The total 2050 stock is one number; the **annual build-out rate** is
what actually constrains a CO₂ supply chain. This section computes the
peak Δ(CO₂)/Δyear over 2024–2050 for each bundle, expresses it in
terms of recognisable CO₂ supply units, and frames it against
representative scales:

* **Climeworks Mammoth (DAC)** ~ 0.036 Mt CO₂/yr nameplate
* **Cement plant CCS** ~ 1 Mt CO₂/yr per facility
* **Large industrial point source** (steel BF, refinery FCC) ~ 2 Mt CO₂/yr
* **EU-27 cement industry total emissions (~2023)** ~ 110 Mt CO₂/yr

These reference scales are intentionally rough — meant to make the
order of magnitude tangible, not to substitute for a detailed
supply-chain model.

In [ ]:
REFERENCE_SCALES_MT = {
    "Climeworks Mammoth (DAC) plants": 0.036,
    "Cement-plant CCS units":           1.0,
    "Large industrial point sources":   2.0,
    "EU-27 cement emissions (2023)":  110.0,
}

ramp_rows = []
for bname, df_b in all_scenarios.items():
    ts = (df_b.groupby("year")["co2_demand_t_per_yr"].sum() / 1e6)  # Mt/yr
    annual_delta = ts.diff().dropna()
    peak_idx = annual_delta.idxmax()
    peak     = float(annual_delta.max())
    avg_24_50 = float(annual_delta.loc[2024:2050].mean())
    total_2050 = float(ts.loc[2050])
    ramp_rows.append({
        "bundle":              bname,
        "Total_2050_Mt":       total_2050,
        "Peak_ramp_Mt_per_yr": peak,
        "Peak_year":           int(peak_idx),
        "Avg_ramp_24_50":      avg_24_50,
    })
ramp = pd.DataFrame(ramp_rows).set_index("bundle")

# Equivalents at peak ramp
for label, scale in REFERENCE_SCALES_MT.items():
    ramp[f"≡ {label} / yr"] = (ramp["Peak_ramp_Mt_per_yr"] / scale).round(1)

print("Annual build-out — peak Δ(CO₂)/Δyear and infrastructure equivalents:")
display(ramp.round(3))

# Visual: trajectory of annual additions
fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, 4.8))

for bname, colour in SCENARIO_COLOURS.items():
    df_b = all_scenarios[bname]
    ts = (df_b.groupby("year")["co2_demand_t_per_yr"].sum() / 1e6)
    annual_delta = ts.diff()
    axA.plot(ts.index, ts.values,        color=colour, lw=2.2, label=bname)
    axB.plot(annual_delta.index, annual_delta.values, color=colour, lw=2.2,
             label=bname)

axA.set_title("Cumulative CO₂ feedstock demand (Mt/yr)")
axA.set_xlim(2019, 2050)
axA.grid(alpha=0.25)
axA.legend(frameon=False, fontsize=9)

axB.set_title("Annual CO₂ build-out rate, Δ(stock)/Δyear (Mt CO₂ added/yr)")
axB.set_xlim(2019, 2050)
axB.grid(alpha=0.25)
# Reference horizontal lines for tangible scales
for label, scale in REFERENCE_SCALES_MT.items():
    if scale > 5:           # only show high-scale references on this plot
        continue
    axB.axhline(scale, color="grey", ls=":", alpha=0.5, lw=0.8)
    axB.text(2020, scale * 1.05, f"{label} (1 unit)",
             fontsize=7, color="grey")

plt.tight_layout()
plt.show()

## 16 — Horizon-side-by-side stacked barplots: usage vs CO₂-side

Following the standard French planning visual (e.g. RTE / SGPE "Consommation
additionnelle" / "Production additionnelle" panels), we lay out **3 horizons
side by side** (2030 / 2035 / 2040 — the action-relevant decade for permitting
and FID) and within each horizon stack:

* **Top row — Usage side (H₂ demand by sector).**  One bar per scenario
  bundle.  All six demand sectors (steel, refinery, ammonia, maritime, olefins,
  eSAF) stacked in consistent colours.  Analogue of the French
  "Consommation additionnelle" reference figure.
* **Bottom row — Feedstock side (CO₂ demand by CO₂-consuming sector).**
  Same scenarios, same horizons, but only the three CO₂-routed sectors
  (maritime e-methanol, olefins MTO, eSAF FT+MtJ).  This is what the H₂
  uptake on the top row *implies* for downstream CO₂ sourcing — the
  "production" / supply problem the H₂ ramp creates.

Reading guide: scenarios are ordered low → high H₂ ambition.  A vertically
growing top stack with a disproportionately growing bottom stack means CO₂
infrastructure is the binding constraint of that scenario.

In [ ]:
# ---------------------------------------------------------------------------
# Horizon-side-by-side stacked barplots
#   Top row : EU-27 H2 demand by sector       (usage side)
#   Bottom  : EU-27 CO2 feedstock by sector   (production-side problem)
#   X-axis  : scenario bundle (low_h2, central, high_h2, industry_stress)
#   Panels  : 2030, 2035, 2040 side-by-side
# ---------------------------------------------------------------------------
HORIZONS       = [2030, 2035, 2040]
BUNDLE_ORDER   = ["low_h2", "central", "high_h2", "industry_stress"]
H2_SECTORS     = ["steel", "refinery", "ammonia", "maritime", "olefins", "esaf"]
CO2_SECTORS_O  = ["maritime", "olefins", "esaf"]

# Robust colour lookup -- fall back to a tab10 cycle if SECTOR_COLOURS is sparse.
try:
    _palette = SECTOR_COLOURS
except NameError:
    _palette = {}
_tab = plt.get_cmap("tab10").colors
_default = {s: _tab[i % 10] for i, s in enumerate(H2_SECTORS)}
SECTOR_C = {s: _palette.get(s, _default[s]) for s in H2_SECTORS}

def _eu27_by_sector(df, year, value_col, sectors):
    """Return Series indexed by sector with EU-27 totals at `year` in Mt."""
    sub = df[(df["year"] == year) & (df["sector"].isin(sectors))]
    return (sub.groupby("sector")[value_col].sum() / 1e6).reindex(sectors).fillna(0.0)

# Pre-aggregate everything so plotting is side-effect free.
h2_panels = {y: pd.DataFrame({b: _eu27_by_sector(all_scenarios[b], y, "h2_demand_t_per_yr", H2_SECTORS)
                              for b in BUNDLE_ORDER}) for y in HORIZONS}
co2_panels = {y: pd.DataFrame({b: _eu27_by_sector(all_scenarios[b], y, "co2_demand_t_per_yr", CO2_SECTORS_O)
                               for b in BUNDLE_ORDER}) for y in HORIZONS}

# Common y-limits per row so bars are visually comparable across horizons.
h2_ymax  = max(df.sum(axis=0).max() for df in h2_panels.values())  * 1.10
co2_ymax = max(df.sum(axis=0).max() for df in co2_panels.values()) * 1.10

fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), sharey="row")

def _stack(ax, df_yb, sectors, colours, ymax, title, ylabel):
    """Stacked bar: rows=sector, cols=bundle.  Annotates each bar with its total."""
    x      = np.arange(df_yb.shape[1])
    bottom = np.zeros(df_yb.shape[1])
    for s in sectors:
        v = df_yb.loc[s].values
        ax.bar(x, v, bottom=bottom, color=colours[s], edgecolor="white",
               linewidth=0.6, label=s)
        bottom = bottom + v
    totals = df_yb.sum(axis=0).values
    for xi, t in zip(x, totals):
        ax.text(xi, t + ymax * 0.012, f"{t:,.1f}", ha="center", va="bottom",
                fontsize=8.5, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(df_yb.columns, rotation=20, ha="right", fontsize=9)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylim(0, ymax)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.set_axisbelow(True)

for j, y in enumerate(HORIZONS):
    _stack(axes[0, j], h2_panels[y],  H2_SECTORS,    SECTOR_C, h2_ymax,
           f"{y}", "H₂ demand  [Mt H₂/yr]" if j == 0 else "")
    _stack(axes[1, j], co2_panels[y], CO2_SECTORS_O, SECTOR_C, co2_ymax,
           f"{y}", "CO₂ feedstock  [Mt CO₂/yr]" if j == 0 else "")

# Row supertitles
fig.text(0.5, 0.965, "Usage side — EU-27 H₂ demand by sector",
         ha="center", fontsize=12.5, fontweight="bold")
fig.text(0.5, 0.495, "Production-side problem — EU-27 CO₂ feedstock by sector",
         ha="center", fontsize=12.5, fontweight="bold")

# Two compact legends (one per row), placed at the right margin
h_handles = [plt.Rectangle((0, 0), 1, 1, fc=SECTOR_C[s]) for s in H2_SECTORS]
c_handles = [plt.Rectangle((0, 0), 1, 1, fc=SECTOR_C[s]) for s in CO2_SECTORS_O]
axes[0, 2].legend(h_handles, H2_SECTORS,    loc="upper left",
                  bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False, title="sector")
axes[1, 2].legend(c_handles, CO2_SECTORS_O, loc="upper left",
                  bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False, title="sector")

fig.tight_layout(rect=[0, 0, 0.92, 0.945])
plt.subplots_adjust(hspace=0.45)
plt.show()

# ---------------------------------------------------------------------------
# Companion table: scenario × horizon totals + CO2:H2 implied ratio.
# ---------------------------------------------------------------------------
rows = []
for y in HORIZONS:
    for b in BUNDLE_ORDER:
        h2  = h2_panels[y][b].sum()
        co2 = co2_panels[y][b].sum()
        rows.append({"year": y, "bundle": b,
                     "H2_Mt": round(h2, 2),
                     "CO2_Mt": round(co2, 2),
                     "CO2:H2_ratio": round(co2 / h2, 2) if h2 > 0 else float("nan")})
print("\nEU-27 totals by horizon × bundle (Mt/yr; CO2:H2 is total-basis):")
print(pd.DataFrame(rows).to_string(index=False))


Interpretation

**Where CO₂ resonates with H₂.** The four-bundle CO₂ total trajectory
(§4 right panel) has the same bundle ordering as the H₂ total (§4 left),
because the CO₂-consuming sectors (maritime e-methanol, olefins MTO,
eSAF) scale jointly with their H₂ demand.

**Where CO₂ diverges from H₂.**

1. **Spread widens.** The bundle-to-bundle CO₂ spread is larger than the
   H₂ spread because the H₂-heavy sectors (ammonia, refinery, steel)
   carry zero CO₂ weight and therefore don't damp the envelope.
2. **Pathway-mix lever is CO₂-amplified.** Going from `high_h2` (40/60
   FT/MtJ) to `industry_stress` (30/70) raises total H₂ by ~5 % but
   total CO₂ by ~10 %, because MtJ is both H₂-heavier *and* CO₂-heavier.
3. **eSAF dominates CO₂.** In every bundle post-2035, eSAF explains
   80–90 % of the total CO₂ feedstock demand. If you are sizing a CO₂
   supply chain for this model, the eSAF sector is effectively *the*
   demand driver.
4. **Stoichiometric ceiling is never breached.** In the §6 scatter, no
   point exceeds the 7.277 t CO₂/t H₂ line — which is the physical
   upper bound for any CO₂-consuming pathway in the registry. This is a
   mass-balance acceptance test, not just visualization.

**How to use this with the main notebook.** Wherever `how_to_use_demandforge.ipynb`
plots `h2_demand_mwh_per_yr`, you can now plot `co2_demand_t_per_yr` with
the same groupby structure (the rows and index are identical). The sector
palette also re-uses cleanly — just remember that steel, refinery, and
ammonia are structural zeros in the CO₂ view.